# 03. Patient-wise Cross-Validation

**Research question:** Does Ridge regression have a lower mean absolute error (MAE) than the training-mean baseline when predicting motor UPDRS scores for patients not seen during training?

**Evaluation plan:** Use only the 33 training patients for cross-validation. In each round, keep all recordings from each patient together and calculate the baseline mean using only that round’s training recordings. Exclude the 9 test patients from cross-validation.

## 1. Import Libraries

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Load the Parkinson’s Telemonitoring Dataset

In [2]:
data_path = Path("../data/parkinsons_updrs.data")
df = pd.read_csv(data_path)

print("Recordings:", len(df))
print("Columns:", df.shape[1])
print("Unique patients:", df["subject#"].nunique())

Recordings: 5875
Columns: 22
Unique patients: 42


## 3. Define Voice Features, Target, and Patient Groups

In [3]:
voice_features = [
    "Jitter(%)", "Jitter(Abs)", "Jitter:RAP", "Jitter:PPQ5", "Jitter:DDP",
    "Shimmer", "Shimmer(dB)", "Shimmer:APQ3", "Shimmer:APQ5",
    "Shimmer:APQ11", "Shimmer:DDA", "NHR", "HNR", "RPDE", "DFA", "PPE"
]

X = df[voice_features]
y = df["motor_UPDRS"]
groups = df["subject#"]

print("Input shape (recordings, features):", X.shape)
print("Target:", y.name)
print("Patient ID column:", groups.name)

Input shape (recordings, features): (5875, 16)
Target: motor_UPDRS
Patient ID column: subject#


## 4. Recreate the Original Patient-wise Train/Test Split

In [4]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]
groups_train = groups.iloc[train_idx]

train_ids = set(groups_train)
test_ids = set(groups.iloc[test_idx])

print("Training patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Patients in both sets:", len(train_ids & test_ids))
print("Test patient IDs:", sorted(map(int, test_ids)))

Training patients: 33
Test patients: 9
Patients in both sets: 0
Test patient IDs: [5, 9, 14, 20, 26, 27, 30, 31, 40]


## 5. Create and Inspect Three Patient-wise Validation Folds

In [5]:
cv = GroupKFold(n_splits=3)
folds = list(cv.split(X_train, y_train, groups=groups_train))

for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
    fit_ids = set(groups_train.iloc[fit_idx])
    val_ids = set(groups_train.iloc[val_idx])

    print(f"Fold {fold}: training patients = {len(fit_ids)}, "
          f"validation patients = {len(val_ids)}, "
          f"overlap = {len(fit_ids & val_ids)}")

Fold 1: training patients = 22, validation patients = 11, overlap = 0
Fold 2: training patients = 22, validation patients = 11, overlap = 0
Fold 3: training patients = 22, validation patients = 11, overlap = 0


**Observation:** In each of the three folds, 22 patients were assigned to training and 11 to validation. No patient appeared in both sets within the same fold. The nine test patients were excluded from cross-validation.

## 6. Calculate the Fold 1 Training Mean

In [6]:
fit_idx, val_idx = folds[0]  # Fold 1
fold1_mean = y_train.iloc[fit_idx].mean()

print(f"Fold 1 training mean: {fold1_mean:.4f}")

Fold 1 training mean: 22.9854


## 7. Evaluate the Mean Baseline on Fold 1

In [7]:
y_val_fold1 = y_train.iloc[val_idx]
baseline_pred_fold1 = pd.Series(fold1_mean, index=y_val_fold1.index)

fold1_baseline_mae = mean_absolute_error(y_val_fold1, baseline_pred_fold1)
print(f"Fold 1 baseline MAE: {fold1_baseline_mae:.4f} points")

Fold 1 baseline MAE: 6.9911 points


## 8. Fit Ridge Regression on Fold 1

In [8]:
ridge_fold1 = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

ridge_fold1.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])

,steps,"[('standardscaler', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None


## 9. Predict Motor UPDRS Scores on Fold 1

In [9]:
ridge_pred_fold1 = ridge_fold1.predict(X_train.iloc[val_idx])

fold1_preview = pd.DataFrame({
    "reference_motor_UPDRS": y_val_fold1,
    "mean_baseline": baseline_pred_fold1,
    "ridge_prediction": ridge_pred_fold1
})

fold1_preview.head(5).round(4)

,reference_motor_UPDRS,mean_baseline,ridge_prediction
438,10.737,22.9854,25.0804
439,11.270,22.9854,22.8807
440,11.803,22.9854,21.8942
441,12.338,22.9854,25.5535
442,12.941,22.9854,23.5265


## 10. Evaluate Ridge Regression on Fold 1

In [10]:
fold1_ridge_mae = mean_absolute_error(y_val_fold1, ridge_pred_fold1)

print(f"Fold 1 baseline MAE: {fold1_baseline_mae:.4f} points")
print(f"Fold 1 Ridge MAE:    {fold1_ridge_mae:.4f} points")

Fold 1 baseline MAE: 6.9911 points
Fold 1 Ridge MAE:    7.0022 points


**Observation:** In Fold 1, the mean baseline had a slightly lower MAE than Ridge regression (6.9911 vs. 7.0022 points). The difference was 0.0111 points.

## 11. Compare the Mean Baseline and Ridge Across All Three Folds

In [11]:
fold_results = []

for fold, (fit_idx, val_idx) in enumerate(folds, start=1):
    X_fit, X_val = X_train.iloc[fit_idx], X_train.iloc[val_idx]
    y_fit, y_val = y_train.iloc[fit_idx], y_train.iloc[val_idx]

    baseline = DummyRegressor(strategy="mean")
    baseline.fit(X_fit, y_fit)
    baseline_pred = baseline.predict(X_val)

    ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    ridge.fit(X_fit, y_fit)
    ridge_pred = ridge.predict(X_val)

    fold_results.append({
        "fold": fold,
        "training_mean": y_fit.mean(),
        "baseline_MAE": mean_absolute_error(y_val, baseline_pred),
        "ridge_MAE": mean_absolute_error(y_val, ridge_pred)
    })

cv_results = pd.DataFrame(fold_results)
cv_results.round(4)

,fold,training_mean,baseline_MAE,ridge_MAE
0,1,22.9854,6.9911,7.0022
1,2,20.2043,8.1967,7.7026
2,3,21.7509,6.6242,9.8480


**Observation:** Ridge had a lower MAE than the mean baseline in Fold 2. The baseline had a lower MAE in Folds 1 and 3. The two methods performed similarly in Fold 1, while the largest difference occurred in Fold 3 (baseline: 6.6242; Ridge: 9.8480 points).

## 12. Calculate the Mean MAE Across Folds

In [12]:
mean_fold_mae = cv_results[["baseline_MAE", "ridge_MAE"]].mean()
print(mean_fold_mae.round(4))

baseline_MAE    7.2706
ridge_MAE       8.1843
dtype: float64


**Observation:** Across three patient-wise validation folds, the mean baseline had a lower mean MAE than Ridge regression (7.2706 vs. 8.1843 points). Ridge performed better in Fold 2, but its MAE was approximately 0.91 points higher on average across folds.

## 13. Save Results for Each Validation Fold

In [13]:
results_path = Path("../results/patient_wise_cv_fold_results.csv")
cv_results.to_csv(results_path, index=False)

print(f"Saved {len(cv_results)} folds to {results_path.name}")

Saved 3 folds to patient_wise_cv_fold_results.csv
